# RQ4 – Feature Importance and Interpretability

**Research Question:** Which input features contribute most to predicting sales revenue, and what insights can be derived from the most influential variables?

**Dataset:** [Kaggle Link](https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

import glob, os

# Auto-detect the dataset file anywhere in the Colab/Kaggle environment
_search_paths = [
    '/kaggle/input/marketing-and-product-performance-dataset/',
    '/content/sample_data/',
    '/content/',
    '/content/drive/MyDrive/',
    '.',
]
_extensions = ['*.csv', '*.xlsx', '*.xls']
DATA_PATH = None

for _dir in _search_paths:
    for _ext in _extensions:
        _matches = glob.glob(os.path.join(_dir, '**', _ext), recursive=True) + glob.glob(os.path.join(_dir, _ext))
        _matches = [m for m in _matches if 'marketing' in m.lower() or 'product' in m.lower() or 'performance' in m.lower()]
        if _matches:
            DATA_PATH = _matches[0]
            break
    if DATA_PATH:
        break

# Final fallback: pick any CSV/Excel in /content
if DATA_PATH is None:
    for _ext in _extensions:
        _all = glob.glob(f'/content/**/{_ext}', recursive=True) + glob.glob(f'./**/{_ext}', recursive=True)
        if _all:
            DATA_PATH = _all[0]
            break

if DATA_PATH:
    print(f'Dataset found: {DATA_PATH}')
else:
    raise FileNotFoundError(
        'Dataset not found. Please upload the CSV/Excel file to Colab via:\n'
        '  Files panel (left sidebar) → Upload, then re-run this cell.')
RANDOM_STATE = 42
TEST_SIZE = 0.2

Dataset found: /content/marketing_and_product_performance 2.csv


In [2]:
# ── Load & Preprocess ─────────────────────────────────────────────────────────
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

candidates = [c for c in df.columns if any(k in c.lower() for k in ['revenue','sales','sale','income','profit','amount'])]
TARGET = candidates[0] if candidates else df.select_dtypes(include=np.number).var().idxmax()
print(f'Target: {TARGET}')

df_clean = df.dropna(thresh=len(df)*0.5, axis=1).copy()
X = df_clean.drop(columns=[TARGET]).copy()
y = df_clean[TARGET]

feature_names = []
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    feature_names.append(col)

X_imp = SimpleImputer(strategy='median').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imp)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

Target: Revenue_Generated


In [3]:
# ── Train Random Forest & Extract Feature Importances ─────────────────────────
rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=False).reset_index(drop=True)
fi_df['Rank'] = fi_df.index + 1

# Estimate direction using Pearson correlation with target
y_aligned = y.reset_index(drop=True)
corr_series = {}
for i, col in enumerate(feature_names):
    col_data = pd.Series(X_imp[:, i])
    corr_val = col_data.corr(y_aligned)
    corr_series[col] = corr_val

fi_df['Direction'] = fi_df['Feature'].map(lambda c: 'Positive' if corr_series.get(c, 0) >= 0 else 'Negative')
fi_df['Importance'] = fi_df['Importance'].round(4)
top10 = fi_df.head(10)
top10

,Feature,Importance,Rank,Direction
0,Customer_ID,0.0781,1,Positive
1,Budget,0.0777,2,Negative
2,Bundle_Price,0.0769,3,Negative
3,Bundle_ID,0.0765,4,Negative
4,Product_ID,0.0760,5,Negative
5,Flash_Sale_ID,0.0758,6,Negative
6,Campaign_ID,0.0756,7,Negative
7,Clicks,0.0752,8,Negative
8,Conversions,0.0747,9,Positive
9,ROI,0.0717,10,Positive


In [4]:
top10[['Rank','Feature','Importance','Direction']].to_csv('tables/RQ4_feature_importance.csv', index=False)
print('Table saved.')

Table saved.


In [5]:
# ── Figure: Top-10 Feature Importance Bar Plot ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

colors = ['#2E4057' if d == 'Positive' else '#E76F51' for d in top10['Direction']]
bars = ax.barh(top10['Feature'][::-1], top10['Importance'][::-1], color=colors[::-1], edgecolor='white', height=0.65)

for bar, val in zip(bars, top10['Importance'][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2E4057', label='Positive correlation'),
                   Patch(facecolor='#E76F51', label='Negative correlation')]
ax.legend(handles=legend_elements, fontsize=10, loc='lower right')

ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Figure 4. Top-10 Feature Importances (Random Forest)\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/RQ4_feature_importance.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


In [6]:
# ── Optional: SHAP (if available) ────────────────────────────────────────────
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test[:200])
    fig2, ax2 = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_values, X_test[:200], feature_names=feature_names, show=False, plot_size=(10,7))
    plt.title('Figure 4b. SHAP Summary Plot', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/RQ4_shap_summary.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('SHAP figure saved.')
except ImportError:
    print('SHAP not installed. Install with: pip install shap')

SHAP figure saved.


## Summary

**RQ4** identifies the top-10 most influential features using Random Forest feature importances, with direction estimated from Pearson correlation. Results in `tables/RQ4_feature_importance.csv` and `figures/RQ4_feature_importance.pdf`.